In [ ]:
# 1. INSTALACIÓN DE LIBRERÍAS ADICIONALES
# Instalamos pypdf para la extracción de texto desde archivos PDF binarios
%pip install pypdf reportlab docling boto3 nltk

In [ ]:
import io
import os
import re
import unicodedata
import boto3
from pypdf import PdfReader
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# 2. DESCARGAS OBLIGATORIAS DE NLTK
# Mantenemos las descargas del laboratorio para el procesamiento de texto
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 3. INTERFAZ DE CARGA DE ARCHIVO
#print("Por favor, sube tu archivo PDF a continuación:")
#archivos_subidos = files.upload()

# Obtener el nombre del archivo cargado dinámicamente
#nombre_archivo = list(archivos_subidos.keys())[0]
#print(f"Archivo cargado con éxito: {nombre_archivo}\n")

# 1. Configuracion de MinIO (Endpoint limpio sin rutas al final)
client = boto3.client("s3", 
                  endpoint_url=os.environ["AWS_ENDPOINT_URL_S3"],
                  aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
                  aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
                 )

bucket_name = "pablo"
object_name = "bdsiniestro_Lima_prueba.pdf"

# 4. DESCARGA Y EXTRACCIÓN DE TEXTO
texto_crudo = ""
try:
    # Traemos el objeto desde el bucket
    respuesta = client.get_object(Bucket=bucket_name, Key=object_name)
    datos_binarios = respuesta["Body"].read()
    respuesta["Body"].close()

    print(f"Descargado desde MinIO: {object_name} ({len(datos_binarios):,} bytes)")

    # Cargamos el binario en memoria, sin escribirlo al disco
    pdf_en_memoria = io.BytesIO(datos_binarios)
    lector_pdf = PdfReader(pdf_en_memoria)

    lista_paginas = []
    for pagina in lector_pdf.pages:
        texto_pagina = pagina.extract_text()
        if texto_pagina:
            lista_paginas.append(texto_pagina)

    texto_crudo = " ".join(lista_paginas)
    print("--- FASE 1: EXTRACCIÓN COMPLETADA ---")
    print(f"Páginas procesadas: {len(lector_pdf.pages)}")
    print(f"Caracteres totales extraídos: {len(texto_crudo)}\n")

except Exception as e:
    print(f"Error al descargar o leer el PDF: {type(e).__name__}: {e}")

# 5. PIPELINE DE PLN
if texto_crudo.strip():
    # A. Normalización Unicode
    texto_normalizado = "".join(
        c for c in unicodedata.normalize('NFD', texto_crudo)
        if unicodedata.category(c) != 'Mn'
    ).lower()

    # B. Limpieza con expresiones regulares
    texto_limpio = re.sub(r'[^a-z\s]', ' ', texto_normalizado)

    # C. Tokenización
    tokens = word_tokenize(texto_limpio)

    # D. Remoción de stopwords
    palabras_vacias = set(stopwords.words('spanish'))
    tokens_finales = [
        token for token in tokens
        if token not in palabras_vacias and len(token) > 1
    ]

    # 6. VISUALIZACIÓN DE LOS RESULTADOS DEL LABORATORIO
    print("--- FASE 2: PIPELINE DE PLN FINALIZADO ---")
    print(f"Muestra del texto limpio:\n{texto_limpio[:300]}...\n")
    print(f"Total de tokens originales : {len(tokens)}")
    print(f"Total sin stopwords        : {len(tokens_finales)}")
    print(f"Vocabulario único          : {len(set(tokens_finales))}\n")

    print("--- PRIMEROS 50 TOKENS ---")
    print(tokens_finales[:50])
else:
    print("No se pudo ejecutar el pipeline: el PDF no contiene texto legible.")


In [ ]:
# 1. INSTALACIÓN
%pip install pypdf boto3 nltk spacy pandas
!python -m spacy download es_core_news_sm

In [ ]:
# 1. INSTALACIÓN DE LIBRERÍAS Y COMPONENTES DE IDIOMA
# Instalamos el lector de PDF y descargamos el modelo en español de spaCy
import io
import os
import re
import unicodedata
import boto3
import pandas as pd
import spacy
from pypdf import PdfReader
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# 2. DESCARGAS OBLIGATORIAS DE NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 3. TRADUCCIÓN DE ETIQUETAS MORFOLÓGICAS
# Los modelos spaCy v3 en español ya no usan tags AnCora en token.tag_.
# La información equivalente está en token.morph (Universal Dependencies).
CATEGORIAS = {
    "VERB": "Verbo", "AUX": "Verbo Auxiliar", "NOUN": "Sustantivo",
    "PROPN": "Sustantivo Propio", "ADJ": "Adjetivo", "PRON": "Pronombre",
    "DET": "Determinante", "ADV": "Adverbio", "ADP": "Preposición",
    "CCONJ": "Conjunción", "SCONJ": "Conjunción Subordinante", "NUM": "Numeral",
    "PART": "Partícula", "INTJ": "Interjección", "PUNCT": "Puntuación",
    "SYM": "Símbolo", "SPACE": "Espacio", "X": "Otro",
}
MODO   = {"Ind": "Indicativo", "Sub": "Subjuntivo", "Imp": "Imperativo", "Cnd": "Condicional"}
FORMA  = {"Fin": "Finito", "Inf": "Infinitivo", "Ger": "Gerundio", "Part": "Participio"}
TIEMPO = {"Pres": "Presente", "Past": "Pasado", "Imp": "Imperfecto", "Fut": "Futuro"}
GENERO = {"Masc": "Masculino", "Fem": "Femenino", "Com": "Común"}
NUMERO = {"Sing": "Singular", "Plur": "Plural"}
PERSONA = {"1": "1ª pers.", "2": "2ª pers.", "3": "3ª pers."}


def _rasgo(token, clave):
    """Devuelve el primer valor de un rasgo morfológico, o None."""
    valores = token.morph.get(clave)
    return valores[0] if valores else None


def mi_pos_espanol(token):
    """Traduce las etiquetas morfológicas de spaCy a texto legible en español."""
    categoria = CATEGORIAS.get(token.pos_, "Otro")

    # Verbos: modo, tiempo, persona y número
    if token.pos_ in ("VERB", "AUX"):
        forma = _rasgo(token, "VerbForm")
        if forma == "Fin":
            detalles = [d for d in (
                MODO.get(_rasgo(token, "Mood"), ""),
                TIEMPO.get(_rasgo(token, "Tense"), ""),
                PERSONA.get(_rasgo(token, "Person"), ""),
                NUMERO.get(_rasgo(token, "Number"), ""),
            ) if d]
            return f"{categoria} ({', '.join(detalles)})" if detalles else categoria
        forma_txt = FORMA.get(forma, "")
        return f"{categoria} ({forma_txt})" if forma_txt else categoria

    # Nominales: género y número
    if token.pos_ in ("NOUN", "PROPN", "ADJ", "DET", "PRON", "NUM"):
        detalles = [d for d in (
            GENERO.get(_rasgo(token, "Gender"), ""),
            NUMERO.get(_rasgo(token, "Number"), ""),
        ) if d]
        return f"{categoria} ({', '.join(detalles)})" if detalles else categoria

    return categoria

# 4. CONEXIÓN A MINIO
client = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_ENDPOINT_URL_S3"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

bucket_name = "pablo"
object_name = "bdsiniestro_Lima_prueba.pdf"

# 5. DESCARGA Y EXTRACCIÓN DE TEXTO
texto_crudo = ""
try:
    respuesta = client.get_object(Bucket=bucket_name, Key=object_name)
    datos_binarios = respuesta["Body"].read()
    respuesta["Body"].close()
    print(f"Descargado desde MinIO: {object_name} ({len(datos_binarios):,} bytes)")

    # El binario se carga en memoria, sin escribir al disco
    lector_pdf = PdfReader(io.BytesIO(datos_binarios))

    lista_paginas = []
    for pagina in lector_pdf.pages:
        texto_pagina = pagina.extract_text()
        if texto_pagina:
            lista_paginas.append(texto_pagina)

    texto_crudo = " ".join(lista_paginas)
    print("--- EXTRAÍDO CORRECTAMENTE ---")
    print(f"Páginas: {len(lector_pdf.pages)} | Caracteres: {len(texto_crudo):,}")

except Exception as e:
    print(f"Error al descargar o leer el PDF: {type(e).__name__}: {e}")


# 6. PIPELINE DE PROCESAMIENTO
if texto_crudo.strip():
    # Normalización y limpieza
    texto_normalizado = "".join(
        c for c in unicodedata.normalize('NFD', texto_crudo)
        if unicodedata.category(c) != 'Mn'
    ).lower()
    texto_limpio = re.sub(r'[^a-z\s]', ' ', texto_normalizado)

    # Tokenización y stopwords
    tokens = word_tokenize(texto_limpio)
    palabras_vacias = set(stopwords.words('spanish'))
    tokens_finales = [t for t in tokens if t not in palabras_vacias and len(t) > 1]

    # --- EXPANSIÓN 1: DISTRIBUCIÓN DE FRECUENCIAS ---
    print("\n=== TABLA DE DISTRIBUCIÓN DE FRECUENCIAS (TOP 20) ===")
    df_frecuencias = pd.Series(tokens_finales).value_counts().reset_index()
    df_frecuencias.columns = ['Token / Palabra', 'Frecuencia Absoluta']
    df_frecuencias['Frecuencia Relativa (%)'] = (
        100 * df_frecuencias['Frecuencia Absoluta'] / len(tokens_finales)
    ).round(2)
    print(df_frecuencias.head(20).to_string(index=False))

    # --- EXPANSIÓN 2: ANÁLISIS MORFOLÓGICO ---
    print("\n=== ANÁLISIS MORFOLÓGICO DETALLADO ===")
    nlp = spacy.load("es_core_news_sm")

    # IMPORTANTE: se usa el texto ORIGINAL, no el normalizado.
    # spaCy necesita las tildes y mayúsculas para etiquetar bien.
    fragmento_original = texto_crudo[:300].strip()
    doc_spacy = nlp(fragmento_original)

    print(f"{'Palabra':<16} | {'POS':<8} | {'Lema':<16} | Categoría detallada")
    print("-" * 82)
    for token in doc_spacy:
        if not token.text.isspace():
            print(f"{token.text:<16} | {token.pos_:<8} | {token.lemma_:<16} | "
                  f"{mi_pos_espanol(token)}")

else:
    print("El archivo PDF no contiene texto procesable.")

In [ ]:
# 1. INSTALACIÓN DE LIBRERÍAS Y MODELOS
!python -m spacy download es_core_news_sm

In [ ]:
import io
import os
import re
import unicodedata
import boto3
import pandas as pd
import spacy
from pypdf import PdfReader
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Descargas iniciales de NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 2. TRADUCCIÓN DE ETIQUETAS MORFOLÓGICAS
# Los modelos spaCy v3 en español no devuelven tags AnCora en token.tag_,
# sino UPOS ("NOUN", "VERB"). El detalle morfológico está en token.morph.
CATEGORIAS = {
    "VERB": "Verbo", "AUX": "Verbo Auxiliar", "NOUN": "Sustantivo",
    "PROPN": "Sustantivo Propio", "ADJ": "Adjetivo", "PRON": "Pronombre",
    "DET": "Determinante", "ADV": "Adverbio", "ADP": "Preposición",
    "CCONJ": "Conjunción", "SCONJ": "Conjunción Subordinante", "NUM": "Numeral",
}
MODO    = {"Ind": "Indicativo", "Sub": "Subjuntivo", "Imp": "Imperativo", "Cnd": "Condicional"}
FORMA   = {"Fin": "Finito", "Inf": "Infinitivo", "Ger": "Gerundio", "Part": "Participio"}
TIEMPO  = {"Pres": "Presente", "Past": "Pasado", "Imp": "Imperfecto", "Fut": "Futuro"}
GENERO  = {"Masc": "Masculino", "Fem": "Femenino", "Com": "Común"}
NUMERO  = {"Sing": "Singular", "Plur": "Plural"}
PERSONA = {"1": "1ª pers.", "2": "2ª pers.", "3": "3ª pers."}

def _rasgo(token, clave):
    valores = token.morph.get(clave)
    return valores[0] if valores else None

def mi_pos_espanol(token):
    cat = CATEGORIAS.get(token.pos_, "Otro")

    if token.pos_ in ("VERB", "AUX"):
        if _rasgo(token, "VerbForm") == "Fin":
            partes = [p for p in (
                MODO.get(_rasgo(token, "Mood"), ""),
                TIEMPO.get(_rasgo(token, "Tense"), ""),
                PERSONA.get(_rasgo(token, "Person"), ""),
                NUMERO.get(_rasgo(token, "Number"), ""),
            ) if p]
            return f"{cat} ({', '.join(partes)})" if partes else cat
        forma = FORMA.get(_rasgo(token, "VerbForm"), "")
        return f"{cat} ({forma})" if forma else cat

    if token.pos_ in ("NOUN", "PROPN", "ADJ", "DET", "PRON", "NUM"):
        partes = [p for p in (
            GENERO.get(_rasgo(token, "Gender"), ""),
            NUMERO.get(_rasgo(token, "Number"), ""),
        ) if p]
        return f"{cat} ({', '.join(partes)})" if partes else cat

    return cat

# 3. CONEXIÓN A MINIO Y EXTRACCIÓN
client = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_ENDPOINT_URL_S3"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

bucket_name = "pablo"
object_name = "bdsiniestro_Lima_prueba.pdf"

texto_crudo = ""
try:
    respuesta = client.get_object(Bucket=bucket_name, Key=object_name)
    datos_binarios = respuesta["Body"].read()
    respuesta["Body"].close()
    print(f"Descargado desde MinIO: {object_name} ({len(datos_binarios):,} bytes)")

    lector_pdf = PdfReader(io.BytesIO(datos_binarios))
    lista_paginas = [p.extract_text() for p in lector_pdf.pages if p.extract_text()]
    texto_crudo = " ".join(lista_paginas)
    print(f"Páginas: {len(lector_pdf.pages)} | Caracteres: {len(texto_crudo):,}")

except Exception as e:
    print(f"Error en la descarga o lectura del binario: {type(e).__name__}: {e}")


# 4. PIPELINE Y CASOS PRÁCTICOS
if texto_crudo.strip():
    texto_normalizado = "".join(
        c for c in unicodedata.normalize('NFD', texto_crudo)
        if unicodedata.category(c) != 'Mn'
    ).lower()
    texto_limpio = re.sub(r'[^a-z\s]', ' ', texto_normalizado)

    tokens = word_tokenize(texto_limpio)
    palabras_vacias = set(stopwords.words('spanish'))
    tokens_finales = [t for t in tokens if t not in palabras_vacias and len(t) > 1]

    # --- CASO PRÁCTICO 1: Frecuencias y exportación ---
    df_frecuencias = pd.Series(tokens_finales).value_counts().reset_index()
    df_frecuencias.columns = ['Token', 'Frecuencia']
    df_frecuencias['Porcentaje'] = (
        100 * df_frecuencias['Frecuencia'] / len(tokens_finales)
    ).round(2)

    nombre_csv = "frecuencias_vocabulario.csv"
    df_frecuencias.to_csv(nombre_csv, index=False, encoding='utf-8')
    print(f"\nCSV guardado localmente: {nombre_csv}")

    # El CSV vuelve al bucket
    try:
        buffer = io.BytesIO()
        df_frecuencias.to_csv(buffer, index=False, encoding='utf-8')
        buffer.seek(0)
        client.put_object(
            Bucket=bucket_name, Key=nombre_csv,
            Body=buffer.getvalue(), ContentType="text/csv",
        )
        print(f"CSV subido a MinIO: s3://{bucket_name}/{nombre_csv}")
    except Exception as e:
        print(f"No se pudo subir el CSV: {type(e).__name__}: {e}")

    print("\n--- TOP 10 PALABRAS MÁS FRECUENTES ---")
    print(df_frecuencias.head(10).to_string(index=False))

    # --- CASO PRÁCTICO 2: Filtrado morfológico selectivo ---
    print("\n--- FILTRADO SELECTIVO: VERBOS Y SUSTANTIVOS PROPIOS ---")
    nlp = spacy.load("es_core_news_sm")

    # Texto ORIGINAL (con tildes y mayúsculas): spaCy las necesita.
    doc_spacy = nlp(texto_crudo[:5000])
    seleccionados = [t for t in doc_spacy if t.pos_ in ("PROPN", "VERB", "AUX")]

    print(f"{'Palabra':<16} | {'Lema':<16} | Análisis morfológico")
    print("-" * 78)
    for token in seleccionados[:40]:
        print(f"{token.text:<16} | {token.lemma_:<16} | {mi_pos_espanol(token)}")

    print(f"\nTotal en el fragmento: {len(seleccionados)} "
          f"({sum(1 for t in seleccionados if t.pos_ == 'PROPN')} propios, "
          f"{sum(1 for t in seleccionados if t.pos_ in ('VERB','AUX'))} verbos)")

else:
    print("El archivo PDF no contiene texto procesable.")

In [ ]:
import re

# TEXTO SUCIO DE PRUEBA (Caso de estudio: Historial clínico / Reporte legal simulado)
corpus_prueba = """
El usuario Juan Perez (ID: 456-A) solicito soporte el dia 24/11/2025.
Su correo es juan.perez@estudio-legal.com y su telefono es +34 611-223-344.
Costo del trámite: $1,550.45 de urgencia.
Visitar la pagina web oficial https://tramites-legales.es para mas informacion.
Texto con errores de formato continuo......   limpiar espacios!!!
"""

print("=== LABORATORIO DE EXPRESIONES REGULARES PARA NLP ===")
print("Texto original de analisis:\n", corpus_prueba)
print("=" * 60)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 1: Extracción de Entidades Numéricas con Estructura Fija (Fechas)
# Uso: Identificar marcas temporales dentro de documentos.
# ---------------------------------------------------------------------
patron_fecha = r'\b\d{2}/\d{2}/\d{4}\b'
fechas_encontradas = re.findall(patron_fecha, corpus_prueba)
print("\n1. Fechas encontradas (Patron: dd/mm/aaaa):")
print(fechas_encontradas)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 2: Extracción de Cuentas de Correo Electrónico
# Uso: Captura automatizada de datos de contacto o anonimización de datos (Data Masking).
# ---------------------------------------------------------------------
patron_email = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
emails_encontrados = re.findall(patron_email, corpus_prueba)
print("\n2. Correos electronicos detectados:")
print(emails_encontrados)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 3: Extracción de Direcciones URL completas
# Uso: Limpieza de enlaces web que actúan como ruido en tareas de clasificación.
# ---------------------------------------------------------------------
patron_url = r'https?://[a-zA-Z0-9.-]+(?:/[a-zA-Z0-9./_-]*)?'
urls_encontradas = re.findall(patron_url, corpus_prueba)
print("\n3. Enlaces web (URLs) identificados:")
print(urls_encontradas)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 4: Extracción de Valores Monetarios (Cifras Financieras)
# Uso: Aislar importes económicos de multas, contratos o presupuestos.
# ---------------------------------------------------------------------
patron_moneda = r'\$[0-9,]+\.[0-9]{2}'
valores_moneda = re.findall(patron_moneda, corpus_prueba)
print("\n4. Valores financieros/moneda detectados:")
print(valores_moneda)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 5: Segmentación / Limpieza Avanzada de Ruido Textual
# Uso: Corregir problemas de espaciado y caracteres repetidos antes de tokenizar.
# ---------------------------------------------------------------------
# A. Reemplazar multiples puntos seguidos por un solo punto espacio
texto_sin_puntos = re.sub(r'\.{2,}', '. ', corpus_prueba)

# B. Eliminar caracteres que no sean letras, espacios o signos basicos de puntuacion
texto_alfabetico = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑ\s]', '', texto_sin_puntos)

# C. Normalizar espacios en blanco duplicados, tabulaciones o saltos de linea
texto_espacios_limpios = re.sub(r'\s+', ' ', texto_alfabetico).strip()

print("\n5. Resultado del texto despues del Pipeline de limpieza con RegEx:")
print(texto_espacios_limpios)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 6: Ejercicio de Anonimización (Caso Práctico de Privacidad)
# Uso: Reemplazar datos sensibles por etiquetas genericas para cumplir leyes de proteccion de datos.
# ---------------------------------------------------------------------
texto_anonimizado = re.sub(patron_email, "[CORREO_ANONIMIZADO]", corpus_prueba)
texto_anonimizado = re.sub(r'\+?\d{2,3}[\s-]?\d{3}[\s-]?\d{3}[\s-]?\d{3}', "[TELEFONO_ANONIMIZADO]", texto_anonimizado)

print("\n6. Documento Anonimizado (Caso Practico de Seguridad):")
print(texto_anonimizado)


In [ ]:
import io
import os
import boto3
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

# Configuración de MinIO
client = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_ENDPOINT_URL_S3"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

bucket_name = "pablo"
object_name = "factura_ejercicio.pdf"

# El PDF se construye en memoria, no en el disco
buffer = io.BytesIO()
doc = SimpleDocTemplate(
    buffer, pagesize=letter,
    rightMargin=30, leftMargin=30, topMargin=30, bottomMargin=30,
    title="Orden de Compra OC-2026-8941",
)
story = []
styles = getSampleStyleSheet()

# Encabezado de la factura / orden de compra
story.append(Paragraph("<b>ORDEN DE COMPRA: OC-2026-8941</b>", styles["Heading1"]))
story.append(Spacer(1, 10))
story.append(Paragraph("<b>Proveedor:</b> Tecnologia Global S.A.", styles["Normal"]))
story.append(Paragraph("<b>Fecha de Emision:</b> 15/09/2026", styles["Normal"]))
story.append(Spacer(1, 15))

# Estructura tabular de los ítems adquiridos
data = [
    ["Codigo / SKU", "Descripcion del Producto", "Cantidad", "Precio Unitario", "Subtotal"],
    ["SKU-9941", "Monitor Asus 24\" ProArt", "2", "USD 250.00", "USD 500.00"],
    ["SKU-1024", "Laptop Lenovo ThinkPad T14", "1", "USD 1200.00", "USD 1200.00"],
    ["SKU-0089", "Teclado Mecanico Logitech G", "5", "USD 80.00", "USD 400.00"],
]

# Estilizado de la tabla para simular un documento comercial
tabla_productos = Table(data, colWidths=[80, 220, 60, 90, 90])
tabla_productos.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2c3e50')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.whitesmoke),
    ('ALIGN', (0,0), (-1,-1), 'LEFT'),
    ('BOTTOMPADDING', (0,0), (-1,0), 6),
    ('BACKGROUND', (0,1), (-1,-1), colors.HexColor('#f8f9fa')),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#bdc3c7')),
]))
story.append(tabla_productos)
story.append(Spacer(1, 15))

# Bloque final de cierre monetario
story.append(Paragraph("<b>MONTO TOTAL GENERAL: USD 2100.00</b>", styles["Normal"]))

# Compilación del documento dentro del buffer
doc.build(story)
buffer.seek(0)
contenido_pdf = buffer.getvalue()
print(f"PDF generado en memoria: {len(contenido_pdf):,} bytes")

# Subida del binario al bucket
try:
    client.put_object(
        Bucket=bucket_name,
        Key=object_name,
        Body=contenido_pdf,
        ContentType="application/pdf",
    )
    print(f"Subido con éxito a MinIO: s3://{bucket_name}/{object_name}")

    # Verificación: consultamos el objeto recién subido
    meta = client.head_object(Bucket=bucket_name, Key=object_name)
    print(f"Verificado en el bucket -> {meta['ContentLength']:,} bytes "
          f"| tipo: {meta['ContentType']} "
          f"| subido: {meta['LastModified']:%Y-%m-%d %H:%M:%S}")

except Exception as e:
    print(f"Error al subir el PDF a MinIO: {type(e).__name__}: {e}")


In [ ]:
import io
import os
import re
import time
import traceback

print("[1/7] Celda iniciada. Cargando librerías...", flush=True)

import boto3
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import DocumentStream

print("      Librerías cargadas correctamente.", flush=True)

BUCKET = "pablo"
OBJETO = "factura_ejercicio.pdf"

def extraer_datos_factura():
    inicio = time.time()

    # --- Paso 2: conexión ---
    print("[2/7] Conectando a MinIO...", flush=True)
    client = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_ENDPOINT_URL_S3"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    )
    print(f"      Endpoint: {os.environ['AWS_ENDPOINT_URL_S3']}", flush=True)

    # --- Paso 3: descarga ---
    print(f"[3/7] Descargando s3://{BUCKET}/{OBJETO} ...", flush=True)
    respuesta = client.get_object(Bucket=BUCKET, Key=OBJETO)
    datos = respuesta["Body"].read()
    respuesta["Body"].close()
    print(f"      Descargados {len(datos):,} bytes.", flush=True)

    if len(datos) == 0:
        print("      AVISO: el objeto está vacío. Se detiene aquí.", flush=True)
        return None

    # --- Paso 4: conversión con Docling ---
    print("[4/7] Convirtiendo con Docling...", flush=True)
    print("      (la primera ejecución descarga modelos, puede tardar varios minutos)",
          flush=True)

    converter = DocumentConverter()
    fuente = DocumentStream(name=OBJETO, stream=io.BytesIO(datos))
    result = converter.convert(fuente)
    texto_documento = result.document.export_to_markdown()

    print(f"      Conversión lista: {len(texto_documento):,} caracteres de Markdown.",
          flush=True)

    # --- Paso 5: vista previa ---
    print("[5/7] Vista previa del texto estructurado:", flush=True)
    print("      " + "-" * 56, flush=True)
    for linea in texto_documento.splitlines()[:12]:
        print(f"      | {linea[:70]}", flush=True)
    print("      " + "-" * 56, flush=True)

    # --- Paso 6: extracción ---
    print("[6/7] Aplicando expresiones regulares...", flush=True)

    patron_oc        = r"ORDEN\s+DE\s+COMPRA:\s*\**\s*(OC-\d{4}-\d{4})"
    patron_proveedor = r"Proveedor:\**\s*([^\n*|]+)"
    patron_fecha     = r"Fecha\s+de\s+Emisi[oó]n:\**\s*(\d{2}/\d{2}/\d{4})"
    patron_total     = r"MONTO\s+TOTAL\s+GENERAL:\s*\**\s*USD\s*([\d,]+\.?\d*)"
    patron_items = (
        r"\|?\s*(SKU-\d+)\s*\|\s*([^\|]+)\|\s*(\d+)\s*"
        r"\|\s*USD\s*([\d,]+\.?\d*)\s*\|\s*USD\s*([\d,]+\.?\d*)"
    )

    match_oc    = re.search(patron_oc, texto_documento)
    match_prov  = re.search(patron_proveedor, texto_documento)
    match_fecha = re.search(patron_fecha, texto_documento)
    match_total = re.search(patron_total, texto_documento)
    items = re.findall(patron_items, texto_documento)

    hallados = sum(1 for x in (match_oc, match_prov, match_fecha, match_total) if x)
    print(f"      Campos de cabecera encontrados: {hallados}/4", flush=True)
    print(f"      Renglones de tabla encontrados: {len(items)}", flush=True)

    if hallados == 0 and not items:
        print("      AVISO: ningún patrón coincidió. Revisa el Markdown de arriba.",
              flush=True)

    # --- Paso 7: resultados ---
    print("[7/7] Resultados\n", flush=True)
    print("=" * 62)
    print(" RESULTADOS DE LA EXTRACCIÓN (PLN + REGEX)")
    print("=" * 62)
    print(f" Código de OC:  {match_oc.group(1) if match_oc else 'No detectado'}")
    print(f" Proveedor:     {match_prov.group(1).strip() if match_prov else 'No detectado'}")
    print(f" Fecha Emisión: {match_fecha.group(1) if match_fecha else 'No detectado'}")
    print(f" Monto Total:   USD {match_total.group(1) if match_total else 'No detectado'}")
    print("-" * 62)

    print(" DETALLE DE ÍTEMS DETECTADOS:")
    if not items:
        print("   (ninguno)")
    suma = 0.0
    for sku, desc, cant, pu, sub in items:
        print(f"   [{sku}] {desc.strip()} -> Cantidad: {cant} "
              f"| Unitario: USD {pu} | Subtotal: USD {sub}")
        suma += float(sub.replace(",", ""))

    print("-" * 62)
    if items and match_total:
        total = float(match_total.group(1).replace(",", ""))
        ok = abs(suma - total) < 0.01
        print(f" Ítems detectados      : {len(items)}")
        print(f" Suma de subtotales    : USD {suma:,.2f}")
        print(f" Total declarado       : USD {total:,.2f}")
        print(f" Validación aritmética : {'CORRECTA' if ok else 'DISCREPANCIA'}")
    else:
        print(" No se pudo validar: faltan ítems o el monto total.")

    print("=" * 62)
    print(f" Tiempo total: {time.time() - inicio:.1f} segundos")
    return texto_documento


# Ejecución con captura de errores: si algo falla, se ve qué y dónde
try:
    markdown = extraer_datos_factura()
    print("\nFIN: la celda terminó correctamente.", flush=True)
except KeyError as e:
    print(f"\nERROR: falta la variable de entorno {e}.", flush=True)
    print("Revisa que AWS_ENDPOINT_URL_S3, AWS_ACCESS_KEY_ID y "
          "AWS_SECRET_ACCESS_KEY estén definidas.", flush=True)
except Exception as e:
    print(f"\nERROR ({type(e).__name__}): {e}\n", flush=True)
    traceback.print_exc()
    print("\nFIN: la celda terminó con error.", flush=True)
